# 1. Load data

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
import xgboost as xgb

In [3]:
df = pd.read_csv("carclaims.csv")

In [4]:
df.shape

(15420, 33)

In [5]:
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange-Claim,NumberOfCars,Year,BasePolicy,FraudFound
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability,No
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision,No
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision,No
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability,No
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision,No


# 2. Clean columns names (cleaning)

In [7]:
df.columns = df.columns.str.lower().str.replace('[^a-z0-9]', '_', regex=True)
print(df.columns.tolist())

['month', 'weekofmonth', 'dayofweek', 'make', 'accidentarea', 'dayofweekclaimed', 'monthclaimed', 'weekofmonthclaimed', 'sex', 'maritalstatus', 'age', 'fault', 'policytype', 'vehiclecategory', 'vehicleprice', 'policynumber', 'repnumber', 'deductible', 'driverrating', 'days_policy_accident', 'days_policy_claim', 'pastnumberofclaims', 'ageofvehicle', 'ageofpolicyholder', 'policereportfiled', 'witnesspresent', 'agenttype', 'numberofsuppliments', 'addresschange_claim', 'numberofcars', 'year', 'basepolicy', 'fraudfound']


# 3. Drop Leakage and Useless Columns

In [9]:
drop_cols = [
    'policynumber',
    'repnumber',
    'daysofpolicyclaimed',
    'dayofweekclaimed',
    'monthclaimed',
    'weekofmonthclaimed',
    'days_policy_claim',
]

drop_cols = [c for c in drop_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print("Remaining shape:", df.shape)

Remaining shape: (15420, 27)


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   month                 15420 non-null  object
 1   weekofmonth           15420 non-null  int64 
 2   dayofweek             15420 non-null  object
 3   make                  15420 non-null  object
 4   accidentarea          15420 non-null  object
 5   sex                   15420 non-null  object
 6   maritalstatus         15420 non-null  object
 7   age                   15420 non-null  int64 
 8   fault                 15420 non-null  object
 9   policytype            15420 non-null  object
 10  vehiclecategory       15420 non-null  object
 11  vehicleprice          15420 non-null  object
 12  deductible            15420 non-null  int64 
 13  driverrating          15420 non-null  int64 
 14  days_policy_accident  15420 non-null  object
 15  pastnumberofclaims    15420 non-null

# Split Data

In [12]:
y = df["fraudfound"] #target fraudfound column
x  = df.drop(columns=["fraudfound"])

X_train, X_other, y_train, y_other = train_test_split(
    x, y, test_size=0.3, stratify=y, random_state=510) #70 - 30 split

X_val, X_test, y_val, y_test = train_test_split(
    X_other, y_other, test_size=0.5, stratify=y_other, random_state=510) #15 - 15 split for validation and test

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

# Encode target to binary (0/1)
y_train = y_train.map({"No": 0, "Yes": 1})
y_val = y_val.map({"No": 0, "Yes": 1})
y_test = y_test.map({"No": 0, "Yes": 1})

# Quick check
print("y_train unique values:", y_train.unique())
print(y_train.value_counts())

Training: (10794, 26) (10794,)
Validation: (2313, 26) (2313,)
Test: (2313, 26) (2313,)
y_train unique values: [0 1]
fraudfound
0    10148
1      646
Name: count, dtype: int64


# 5. Handle Missing Values & Fix Errors


In [14]:
df[df["age"] == 0]["fraudfound"].value_counts()

fraudfound
No     289
Yes     31
Name: count, dtype: int64

In [15]:
(df["age"] == 0).sum()

np.int64(320)

In [16]:
# Create flag before changing age
X_train["is_age_placeholder"] = (X_train["age"] == 0).astype(int)
X_test["is_age_placeholder"] = (X_test["age"] == 0).astype(int)

# Convert invalid 0 ages to NaN
X_train["age"] = X_train["age"].replace(0, np.nan)
X_test["age"] = X_test["age"].replace(0, np.nan)

# Compute medians from TRAIN only
group_medians = X_train.groupby(["maritalstatus", "vehicleprice"])["age"].median()
global_median = X_train["age"].median()

# Fill TRAIN
train_keys = pd.MultiIndex.from_frame(X_train[["maritalstatus", "vehicleprice"]])
X_train["age"] = X_train["age"].fillna(
    pd.Series(train_keys.map(group_medians), index=X_train.index)
)
X_train["age"] = X_train["age"].fillna(global_median)


In [17]:
# Fill TEST using TRAIN medians
test_keys = pd.MultiIndex.from_frame(X_test[["maritalstatus", "vehicleprice"]])
X_test["age"] = X_test["age"].fillna(
    pd.Series(test_keys.map(group_medians), index=X_test.index)
)
X_test["age"] = X_test["age"].fillna(global_median)

In [18]:
# Check
print((X_train["age"] == 0).sum())
print((X_test["age"] == 0).sum())

0
0


# 6. Feature engineering (new columns)

In [20]:
#Applied separately to X_train, X_val, X_test

# High claim history
X_train["high_claim_history"] = X_train["pastnumberofclaims"].isin(
    ["2 to 4", "more than 4"]
).astype(int)

X_val["high_claim_history"] = X_val["pastnumberofclaims"].isin(
    ["2 to 4", "more than 4"]
).astype(int)

X_test["high_claim_history"] = X_test["pastnumberofclaims"].isin(
    ["2 to 4", "more than 4"]
).astype(int)


# Weekend accident
X_train["weekend_accident"] = X_train["dayofweek"].isin(
    ["Saturday", "Sunday"]
).astype(int)

X_val["weekend_accident"] = X_val["dayofweek"].isin(
    ["Saturday", "Sunday"]
).astype(int)

X_test["weekend_accident"] = X_test["dayofweek"].isin(
    ["Saturday", "Sunday"]
).astype(int)


# Old vehicle
X_train["old_vehicle"] = X_train["ageofvehicle"].isin(
    ["7 years", "more than 7"]
).astype(int)

X_val["old_vehicle"] = X_val["ageofvehicle"].isin(
    ["7 years", "more than 7"]
).astype(int)

X_test["old_vehicle"] = X_test["ageofvehicle"].isin(
    ["7 years", "more than 7"]
).astype(int)


# High deductible
X_train["high_deductible"] = (X_train["deductible"] > 400).astype(int)
X_val["high_deductible"] = (X_val["deductible"] > 400).astype(int)
X_test["high_deductible"] = (X_test["deductible"] > 400).astype(int)


# Multiple cars
X_train["multiple_cars"] = (X_train["numberofcars"] != "1 vehicle").astype(int)
X_val["multiple_cars"] = (X_val["numberofcars"] != "1 vehicle").astype(int)
X_test["multiple_cars"] = (X_test["numberofcars"] != "1 vehicle").astype(int)

# 7. Encode categoricals

### Ordinal Encoding

In [23]:
ordinal_cols = [
    "vehicleprice",
    "days_policy_accident",
    "pastnumberofclaims",
    "ageofvehicle",
    "ageofpolicyholder",
    "numberofsuppliments",
    "addresschange_claim",
    "numberofcars"
]


In [24]:
ord_enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[ordinal_cols] = ord_enc.fit_transform(X_train[ordinal_cols])
X_val[ordinal_cols] = ord_enc.transform(X_val[ordinal_cols])
X_test[ordinal_cols] = ord_enc.transform(X_test[ordinal_cols])

### Label Encoding

In [26]:
label_cols = [
    "accidentarea",
    "sex",
    "fault",
    "policereportfiled",
    "witnesspresent",
    "agenttype",
    "basepolicy"
]

# Scale numerical variables


In [27]:
label_encoders = {}

for col in label_cols:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_val[col] = le.transform(X_val[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

    label_encoders[col] = le

### One-hot (nominal)


In [29]:
onehot_cols = [
    "make",
    "policytype",
    "vehiclecategory",
    "month",
    "dayofweek",
    "maritalstatus"
]

In [30]:
# Train
X_train = pd.get_dummies(X_train, columns=onehot_cols, drop_first=True)

# Val/Test
X_val = pd.get_dummies(X_val, columns=onehot_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=onehot_cols, drop_first=True)

# Align columns (CRUCIAL)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# 8. Scale numerical variables

In [32]:
num_cols = ["weekofmonth", "age", "deductible", "driverrating", "year"]
scaler = StandardScaler()

# Fit on TRAIN only
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# Apply to val and test
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train.head()

,weekofmonth,accidentarea,sex,age,fault,vehicleprice,deductible,driverrating,days_policy_accident,pastnumberofclaims,...,month_Sep,dayofweek_Monday,dayofweek_Saturday,dayofweek_Sunday,dayofweek_Thursday,dayofweek_Tuesday,dayofweek_Wednesday,maritalstatus_Married,maritalstatus_Single,maritalstatus_Widow
7867,1.727909,1,1,-1.029873,1,1.0,-0.172803,-1.337382,3.0,1.0,...,False,False,False,False,True,False,False,True,False,False
13654,0.167609,1,1,-1.029873,0,4.0,-0.172803,0.453093,3.0,0.0,...,False,False,False,False,True,False,False,True,False,False
8819,-1.392692,1,1,0.687445,0,1.0,-0.172803,1.348330,3.0,1.0,...,False,True,False,False,False,False,False,True,False,False
723,0.947759,1,1,-0.539210,0,0.0,-0.172803,0.453093,3.0,0.0,...,False,False,False,False,False,False,True,False,True,False
14853,0.947759,1,1,0.932776,0,0.0,-0.172803,1.348330,3.0,1.0,...,False,False,True,False,False,False,False,True,False,False


# Resampling


### No Resampling

In [35]:
def no_resampling(X_train, y_train):
    """
    Return original training data without modification
    """
    return X_train, y_train

### Random Oversampling

In [37]:
def random_oversampling(X_train, y_train):
    """
    Apply random oversampling to balance classes
    """
    ros = RandomOverSampler(random_state=42)
    X_resampled, y_resampled = ros.fit_resample(X_train, y_train)
    
    return X_resampled, y_resampled

### SMOTE

In [39]:
def smote_resampling(X_train, y_train):
    """
    Apply SMOTE to generate synthetic minority samples
    """
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    
    return X_resampled, y_resampled

# Logistic Regression

# Random Forest

# XGBoost

In [43]:
def build_xgb_model(scale_pos_weight=1.0):
    # Core XGBoost model
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    )
    
    return model

###  XGB-1: no resampling

In [45]:
# XGB-1: no resampling

# Keep original training data
X_train_nr, y_train_nr = no_resampling(X_train, y_train)

# Compute imbalance ratio
neg = (y_train_nr == 0).sum()
pos = (y_train_nr == 1).sum()
scale_pos_weight = neg / pos

# Build model
xgb_xgb1 = build_xgb_model(scale_pos_weight=scale_pos_weight)

# Fit model
xgb_xgb1.fit(X_train_nr, y_train_nr)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [46]:
# XGB-1: predictions and metrics

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

# Validation predictions
y_val_proba_xgb1 = xgb_xgb1.predict_proba(X_val)[:, 1]
y_val_pred_xgb1 = (y_val_proba_xgb1 >= 0.5).astype(int)

# Validation metrics
xgb1_val_pr_auc = average_precision_score(y_val, y_val_proba_xgb1)
xgb1_val_roc_auc = roc_auc_score(y_val, y_val_proba_xgb1)
xgb1_val_precision = precision_score(y_val, y_val_pred_xgb1, zero_division=0)
xgb1_val_recall = recall_score(y_val, y_val_pred_xgb1, zero_division=0)
xgb1_val_f1 = f1_score(y_val, y_val_pred_xgb1, zero_division=0)
xgb1_val_cm = confusion_matrix(y_val, y_val_pred_xgb1)

print("XGB-1 Validation")
print(f"PR-AUC:   {xgb1_val_pr_auc:.4f}")
print(f"ROC-AUC:  {xgb1_val_roc_auc:.4f}")
print(f"Precision:{xgb1_val_precision:.4f}")
print(f"Recall:   {xgb1_val_recall:.4f}")
print(f"F1-score: {xgb1_val_f1:.4f}")
print("Confusion matrix:")
print(xgb1_val_cm)

# Test predictions
y_test_proba_xgb1 = xgb_xgb1.predict_proba(X_test)[:, 1]
y_test_pred_xgb1 = (y_test_proba_xgb1 >= 0.5).astype(int)

# Test metrics
xgb1_test_pr_auc = average_precision_score(y_test, y_test_proba_xgb1)
xgb1_test_roc_auc = roc_auc_score(y_test, y_test_proba_xgb1)
xgb1_test_precision = precision_score(y_test, y_test_pred_xgb1, zero_division=0)
xgb1_test_recall = recall_score(y_test, y_test_pred_xgb1, zero_division=0)
xgb1_test_f1 = f1_score(y_test, y_test_pred_xgb1, zero_division=0)
xgb1_test_cm = confusion_matrix(y_test, y_test_pred_xgb1)

print("\nXGB-1 Test")
print(f"PR-AUC:   {xgb1_test_pr_auc:.4f}")
print(f"ROC-AUC:  {xgb1_test_roc_auc:.4f}")
print(f"Precision:{xgb1_test_precision:.4f}")
print(f"Recall:   {xgb1_test_recall:.4f}")
print(f"F1-score: {xgb1_test_f1:.4f}")
print("Confusion matrix:")
print(xgb1_test_cm)

XGB-1 Validation
PR-AUC:   0.2608
ROC-AUC:  0.8481
Precision:0.1748
Recall:   0.7986
F1-score: 0.2868
Confusion matrix:
[[1650  524]
 [  28  111]]

XGB-1 Test
PR-AUC:   0.2410
ROC-AUC:  0.8632
Precision:0.1744
Recall:   0.8188
F1-score: 0.2875
Confusion matrix:
[[1640  535]
 [  25  113]]


### XGB-2: Random Oversampling

In [48]:
# XGB-2: random oversampling

# Resample training data
X_train_ros, y_train_ros = random_oversampling(X_train, y_train)

# Build model
xgb_xgb2 = build_xgb_model(scale_pos_weight=1.0)

# Fit model
xgb_xgb2.fit(X_train_ros, y_train_ros)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [49]:
# XGB-2: predictions and metrics

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

# Validation predictions
y_val_proba_xgb2 = xgb_xgb2.predict_proba(X_val)[:, 1]
y_val_pred_xgb2 = (y_val_proba_xgb2 >= 0.5).astype(int)

# Validation metrics
xgb2_val_pr_auc = average_precision_score(y_val, y_val_proba_xgb2)
xgb2_val_roc_auc = roc_auc_score(y_val, y_val_proba_xgb2)
xgb2_val_precision = precision_score(y_val, y_val_pred_xgb2, zero_division=0)
xgb2_val_recall = recall_score(y_val, y_val_pred_xgb2, zero_division=0)
xgb2_val_f1 = f1_score(y_val, y_val_pred_xgb2, zero_division=0)
xgb2_val_cm = confusion_matrix(y_val, y_val_pred_xgb2)

print("XGB-2 Validation")
print(f"PR-AUC:   {xgb2_val_pr_auc:.4f}")
print(f"ROC-AUC:  {xgb2_val_roc_auc:.4f}")
print(f"Precision:{xgb2_val_precision:.4f}")
print(f"Recall:   {xgb2_val_recall:.4f}")
print(f"F1-score: {xgb2_val_f1:.4f}")
print("Confusion matrix:")
print(xgb2_val_cm)

# Test predictions
y_test_proba_xgb2 = xgb_xgb2.predict_proba(X_test)[:, 1]
y_test_pred_xgb2 = (y_test_proba_xgb2 >= 0.5).astype(int)

# Test metrics
xgb2_test_pr_auc = average_precision_score(y_test, y_test_proba_xgb2)
xgb2_test_roc_auc = roc_auc_score(y_test, y_test_proba_xgb2)
xgb2_test_precision = precision_score(y_test, y_test_pred_xgb2, zero_division=0)
xgb2_test_recall = recall_score(y_test, y_test_pred_xgb2, zero_division=0)
xgb2_test_f1 = f1_score(y_test, y_test_pred_xgb2, zero_division=0)
xgb2_test_cm = confusion_matrix(y_test, y_test_pred_xgb2)

print("\nXGB-2 Test")
print(f"PR-AUC:   {xgb2_test_pr_auc:.4f}")
print(f"ROC-AUC:  {xgb2_test_roc_auc:.4f}")
print(f"Precision:{xgb2_test_precision:.4f}")
print(f"Recall:   {xgb2_test_recall:.4f}")
print(f"F1-score: {xgb2_test_f1:.4f}")
print("Confusion matrix:")
print(xgb2_test_cm)

XGB-2 Validation
PR-AUC:   0.2806
ROC-AUC:  0.8556
Precision:0.1716
Recall:   0.8273
F1-score: 0.2843
Confusion matrix:
[[1619  555]
 [  24  115]]

XGB-2 Test
PR-AUC:   0.2526
ROC-AUC:  0.8645
Precision:0.1669
Recall:   0.8406
F1-score: 0.2785
Confusion matrix:
[[1596  579]
 [  22  116]]


### XGB-3: SMOTE

In [51]:
# XGB-3: SMOTE

# Resample training data with SMOTE
X_train_smote, y_train_smote = smote_resampling(X_train, y_train)

# Build model
xgb_xgb3 = build_xgb_model(scale_pos_weight=1.0)

# Fit model
xgb_xgb3.fit(X_train_smote, y_train_smote)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [52]:
# XGB-3: predictions and metrics

# Validation predictions
y_val_proba_xgb3 = xgb_xgb3.predict_proba(X_val)[:, 1]
y_val_pred_xgb3 = (y_val_proba_xgb3 >= 0.5).astype(int)

# Validation metrics
xgb3_val_pr_auc = average_precision_score(y_val, y_val_proba_xgb3)
xgb3_val_roc_auc = roc_auc_score(y_val, y_val_proba_xgb3)
xgb3_val_precision = precision_score(y_val, y_val_pred_xgb3, zero_division=0)
xgb3_val_recall = recall_score(y_val, y_val_pred_xgb3, zero_division=0)
xgb3_val_f1 = f1_score(y_val, y_val_pred_xgb3, zero_division=0)
xgb3_val_cm = confusion_matrix(y_val, y_val_pred_xgb3)

print("XGB-3 Validation")
print(f"PR-AUC:   {xgb3_val_pr_auc:.4f}")
print(f"ROC-AUC:  {xgb3_val_roc_auc:.4f}")
print(f"Precision:{xgb3_val_precision:.4f}")
print(f"Recall:   {xgb3_val_recall:.4f}")
print(f"F1-score: {xgb3_val_f1:.4f}")
print("Confusion matrix:")
print(xgb3_val_cm)

# Test predictions
y_test_proba_xgb3 = xgb_xgb3.predict_proba(X_test)[:, 1]
y_test_pred_xgb3 = (y_test_proba_xgb3 >= 0.5).astype(int)

# Test metrics
xgb3_test_pr_auc = average_precision_score(y_test, y_test_proba_xgb3)
xgb3_test_roc_auc = roc_auc_score(y_test, y_test_proba_xgb3)
xgb3_test_precision = precision_score(y_test, y_test_pred_xgb3, zero_division=0)
xgb3_test_recall = recall_score(y_test, y_test_pred_xgb3, zero_division=0)
xgb3_test_f1 = f1_score(y_test, y_test_pred_xgb3, zero_division=0)
xgb3_test_cm = confusion_matrix(y_test, y_test_pred_xgb3)

print("\nXGB-3 Test")
print(f"PR-AUC:   {xgb3_test_pr_auc:.4f}")
print(f"ROC-AUC:  {xgb3_test_roc_auc:.4f}")
print(f"Precision:{xgb3_test_precision:.4f}")
print(f"Recall:   {xgb3_test_recall:.4f}")
print(f"F1-score: {xgb3_test_f1:.4f}")
print("Confusion matrix:")
print(xgb3_test_cm)

XGB-3 Validation
PR-AUC:   0.2670
ROC-AUC:  0.8474
Precision:0.6111
Recall:   0.0791
F1-score: 0.1401
Confusion matrix:
[[2167    7]
 [ 128   11]]

XGB-3 Test
PR-AUC:   0.2232
ROC-AUC:  0.8570
Precision:0.2727
Recall:   0.0217
F1-score: 0.0403
Confusion matrix:
[[2167    8]
 [ 135    3]]
